# Baseline Model and Validation Strategy

In this notebook we establish a validation framework, and evaluate a naive baseline model.
Model is located in ```src/models/``` and returns sales of item in shop from previous month or mean of the sales of this item at this shop from all the previous months and 0 if this item was never sold at this shop.

---

## 1. Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import root_mean_squared_error

from src.data.etl import (
    ItemCategoryLoader,
    ItemsLoader,
    ShopsLoader,
    TrainLoader,
    DfFinalLoader,
    ForSubmissionLoader,
)

from src.models.baseline_model import BaseLineModel

np.random.seed(42)


## 2. Data loading

In [2]:
project_root = Path.cwd().parent
raw_path = project_root / "data" / "raw"
processed_path = project_root / "data" / "processed"

items = ItemsLoader(raw_data_path=raw_path, cache_path=processed_path).load()
item_cats = ItemCategoryLoader(raw_data_path=raw_path, cache_path=processed_path).load()
shops = ShopsLoader(raw_data_path=raw_path, cache_path=processed_path).load()
train = ShopsLoader(raw_data_path=raw_path, cache_path=processed_path).load()
df = DfFinalLoader(cache_path=processed_path, shops=shops, items=items, item_cats=item_cats, train=train).load()
subm = ForSubmissionLoader(raw_data_path=raw_path, cache_path=processed_path, shops=shops, items=items).load()

## 3. Creating Validation dataset

First, let's identify the maximum `date_block_num` (month index) available in our dataset.

In [3]:
df["date_block_num"].max()

np.int64(33)

Assuming `date_block_num` goes up to 33, we will use months `< 32` for training and month `32` for validation. Month `33` corresponds to the test set.

In [4]:
Train = df[df["date_block_num"] < 32]
Valid = df[df["date_block_num"] == 32][['corrected_shop_id', 'corrected_item_id', 'item_cnt_day']]

Since our objective is to predict monthly sales, we group our validation set by shop and item, summing up the daily sales.

In [5]:
columns_to_group = list(Valid.columns.drop('item_cnt_day'))
Valid = (
    Valid
    .groupby(columns_to_group, as_index=False)
    .agg(item_cnt_day=("item_cnt_day", "sum"))
)

We need to understand how many shop-item combinations present in the test (submission) set were actually seen during training.

In [6]:
subm_pairs = set(map(tuple, subm[['corrected_shop_id', 'corrected_item_id']].drop_duplicates().values))
train_pairs = set(map(tuple, Train[ ['corrected_shop_id', 'corrected_item_id']].drop_duplicates().values))
intersection_train = subm_pairs & train_pairs


print(f'Number of pairs in submission: {len(subm_pairs)}')
print(f'Number of pairs in train: {len(train_pairs)}')
print(f'Number of intersecting pairs between Train и submission: {len(intersection_train)}')

print(f'Proportion of pairs in submission included in Train: {len(intersection_train) / len(subm_pairs):.2%}')
print(f'Proportion of pairs in Train included in submission: {len(intersection_train) / len(train_pairs):.2%}')

Number of pairs in submission: 214116
Number of pairs in train: 405548
Number of intersecting pairs between Train и submission: 101187
Proportion of pairs in submission included in Train: 47.26%
Proportion of pairs in Train included in submission: 24.95%


Calculating the number of existing pairs in the validation set compared to the maximum possible pairs (Cartesian product of all validation shops and items).

In [7]:
valid_pairs_count = Valid[['corrected_shop_id', 'corrected_item_id']].drop_duplicates().shape[0]
possible_valid_pairs_count = Valid['corrected_shop_id'].nunique() * Valid['corrected_item_id'].nunique()

print('Number of pairs in Valid:', valid_pairs_count)
print('Possible pairs in Valid:', possible_valid_pairs_count)
print('Completion:', valid_pairs_count / possible_valid_pairs_count)

Number of pairs in Valid: 29676
Possible pairs in Valid: 218569
Completion: 0.13577405762024808


Next, we check the intersection of our current valid set with the training set.

In [8]:
valid_pairs = set(map(tuple, Valid[['corrected_shop_id', 'corrected_item_id']].drop_duplicates().values))
intersection_valid = train_pairs & valid_pairs

print(f'Number of pairs in Valid: {len(valid_pairs)}')
print(f'Number of intersecting pairs between Valid and Train: {len(intersection_valid)}')

print(f'Proportion of pairs in Valid included in Train: {len(intersection_valid) / len(valid_pairs):.2%}')
print(f'Proportion of pairs in Train included in Valid: {len(intersection_valid) / len(train_pairs):.2%}')

Number of pairs in Valid: 29676
Number of intersecting pairs between Valid and Train: 24517
Proportion of pairs in Valid included in Train: 82.62%
Proportion of pairs in Train included in Valid: 6.05%


The test set requires predictions for new pairs. To make our validation scheme reliable and representative of the test environment, we inject artificial zero-sales records by randomly sampling pairs from the missing subset.

In [9]:
valid_shops = set(Valid['corrected_shop_id'].unique())
valid_items = set(Valid['corrected_item_id'].unique())
possible_valid_pairs = {(shop, item) for shop in valid_shops for item in valid_items}
not_in_valid_pairs = list(possible_valid_pairs - valid_pairs)
idx = np.random.choice(len(not_in_valid_pairs), size=int(len(valid_pairs) * 4), replace=False)
sample_pairs = np.array(not_in_valid_pairs)[idx]

new_rows = pd.DataFrame(
    sample_pairs,
    columns=['corrected_shop_id', 'corrected_item_id']
)
new_rows['item_cnt_day'] = 0

Valid = pd.concat([Valid, new_rows], ignore_index=True)

Re-evaluating the intersections after extending the validation set:

In [10]:
new_valid_pairs = set(map(tuple, Valid[['corrected_shop_id', 'corrected_item_id']].drop_duplicates().values))
intersection_new_valid = train_pairs & new_valid_pairs

print(f'Number of pairs in Valid: {len(new_valid_pairs)}')
print(f'Number of intersecting pairs between Valid and Train: {len(intersection_new_valid)}')

print(f'Proportion of pairs in Valid included in Train: {len(intersection_new_valid) / len(new_valid_pairs):.2%}')
print(f'Proportion of pairs in Train included in Valid: {len(intersection_new_valid) / len(train_pairs):.2%}')

Number of pairs in Valid: 148380
Number of intersecting pairs between Valid and Train: 85879
Proportion of pairs in Valid included in Train: 57.88%
Proportion of pairs in Train included in Valid: 21.18%


## 4. Creating model

We fit a simple baseline model on the training data and predict on our modified validation set.

In [11]:
model = BaseLineModel()
model.fit(Train)
true_ans = Valid['item_cnt_day']
Valid = Valid.drop(columns=['item_cnt_day'])
ans = model.predict(Valid)

Viewing the baseline predictions and true answers:

In [12]:
ans

0         0.0
1         0.0
2         2.0
3         0.0
4         0.0
         ... 
148375    0.0
148376    0.0
148377    0.0
148378    0.0
148379    0.0
Name: item_cnt_day, Length: 148380, dtype: float64

In [13]:
true_ans

0         1.0
1         1.0
2         1.0
3         1.0
4         1.0
         ... 
148375    0.0
148376    0.0
148377    0.0
148378    0.0
148379    0.0
Name: item_cnt_day, Length: 148380, dtype: float64

## 5. Metrics

Evaluating the exact match accuracy overall, and isolated by zero and non-zero targets.

In [14]:
len(true_ans[true_ans == ans]) / len(true_ans)

0.7582490901738779

In [15]:
len(true_ans[(true_ans == ans) & (true_ans != 0.0)]) / len(true_ans[true_ans != 0.0])

0.18422384986667567

In [16]:
len(true_ans[(true_ans == ans) & (true_ans == 0.0)]) / len(true_ans[true_ans == 0.0])

0.9014593315537291

Computing Root Mean Squared Error (RMSE) for our baseline model and comparing it against naive constants.

In [17]:
root_mean_squared_error(true_ans, ans)

7.553108457110487

In [18]:
root_mean_squared_error(true_ans, np.ones(len(true_ans)))

8.307250189747263

In [19]:
root_mean_squared_error(true_ans, np.zeros(len(true_ans)))

8.306145161610614

In [20]:
root_mean_squared_error(true_ans, np.full((len(true_ans), 1), true_ans.mean()))

8.291630860324684